This code will tokenize Pile (https://huggingface.co/datasets/monology/pile-uncopyrighted) and Lymsys (https://huggingface.co/datasets/lmsys/lmsys-chat-1m) datasets for Qwen - (For ~400 mil tokens) and upload them to HF for training 

In [ ]:
from datasets import load_dataset, concatenate_datasets, Dataset
from tqdm import tqdm  # Progress bar

# Load datasets with streaming enabled
pile_streamed = load_dataset("monology/pile-uncopyrighted", split="train", streaming=True)
lmsys_streamed = load_dataset("science-of-finetuning/lmsys-chat-1m-chat-formatted", split="train", streaming=True)

# Process pile: rename and add source 
def process_pile(example):
    return {"text": example["text"], "dataset": "pile"}

def process_lmsys(example):
    return {"text": example["text_qwen2_5"], "dataset": "lmsys"}

# Apply transforms and take first 500k examples with progress bars
pile_processed = map(process_pile, pile_streamed)
lmsys_processed = map(process_lmsys, lmsys_streamed)

# Convert to in-memory Dataset (limit to 500k examples each) with tqdm progress bar
pile_list = [x for x in tqdm(pile_processed, total=500_000, desc="Loading Pile", unit=" examples")][:500_000]
lmsys_list = [x for x in tqdm(lmsys_processed, total=500_000, desc="Loading LMSYS", unit=" examples")][:500_000]

# Convert to Hugging Face Dataset objects
pile_ds = Dataset.from_list(pile_list)
lmsys_ds = Dataset.from_list(lmsys_list)

# Combine the datasets
combined_dataset = concatenate_datasets([pile_ds, lmsys_ds])

# Upload to Hugging Face Hub
hf_repo_id = "AndrisWillow/pile-lmsys_qwen_format-mix-1m"
combined_dataset.push_to_hub(hf_repo_id)


In [ ]:
# Now we use the combined dataset

from sae_lens import PretokenizeRunner, PretokenizeRunnerConfig

cfg = PretokenizeRunnerConfig(
    tokenizer_name="meta-llama/Llama-3.2-1B", # Uses AutoTokenizer from HF
    dataset_path="AndrisWillow/pile-lmsys-mix-1m",
    shuffle=True,
    num_proc=32,  # increase this number depending on how many CPUs you have
    context_size=1024, 
    begin_batch_token=None,
    begin_sequence_token=None, #
    sequence_separator_token=None,
    hf_repo_id="AndrisWillow/Pile-Lmsys-1m-tokenized-Llama-3.2-1B",
)

dataset = PretokenizeRunner(cfg).run()